<a href="https://colab.research.google.com/github/profliuhao/CSIT599/blob/main/CSIT599_Online_lab6_transformer_nmt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6 — Building a Transformer for Neural Machine Translation (Keras)

**Module 6: BERT, GPT, and Large Language Models**

Last module you saw what attention buys a recurrent translator. This module's readings
introduce BERT and GPT — and both are stacks of exactly one thing: the **transformer**
(Vaswani et al., 2017). In this lab you assemble the complete architecture yourself in
Keras and train it on the same English → German task as Lab 5:

```
Encoder:  [Embedding + Positional Encoding] -> N x [Self-Attention -> FFN]
Decoder:  [Embedding + Positional Encoding] -> N x [Masked Self-Attn -> Cross-Attn -> FFN]
```

BERT is this encoder stack; GPT is this decoder stack. Build it once and you can read
every LLM architecture diagram for the rest of your career.

## What you will do
Fill in each `___BLANK___` (a hint comment sits next to every one):
  1. the **sinusoidal positional encoding** (the sine/cosine formula)
  2. **scaled dot-product attention** (QKᵀ, masking, softmax)
  3. the **residual connections** inside the encoder layer
  4. the **learning rate** in the training setup

The multi-head attention layer, feed-forward network, decoder layer, masks, training
loop, and translation function are provided — read them as you go: they are the same
components you built by hand, now assembled into the full architecture.

## How to use this file
* Use **Google Colab with a GPU runtime** (`Runtime > Change runtime type > T4 GPU`).
  Training takes ~5-10 minutes on GPU.
* Fill in each `___BLANK___`, then run the cells from top to bottom (or "Run All").

## Setup — imports and hyperparameters

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import urllib.request
import zipfile
import os
import re
from tqdm.notebook import tqdm

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# ==============================================================================
# HYPERPARAMETERS
# ==============================================================================
BATCH_SIZE = 128
D_MODEL = 256          # Model dimension (embedding size per token)
NUM_HEADS = 8          # Attention heads (parallel attention subspaces)
NUM_LAYERS = 2         # Encoder/decoder layers (network depth)
D_FF = 256             # Inner dimension of the feed-forward network
DROPOUT_RATE = 0.1     # Dropout on embeddings, attention, and FFN outputs
MAX_LENGTH = 20        # Pad/truncate every sentence to this many tokens
EPOCHS = 10

## Data loading *(provided — same pipeline as Lab 5)*

In [ ]:
def download_data():
    """Download the English-German translation dataset if not already present."""
    url = "http://www.manythings.org/anki/deu-eng.zip"
    filename = "deu-eng.zip"
    if not os.path.exists("deu.txt"):
        print("Downloading dataset...")
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"}
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req) as response, open(filename, "wb") as out_file:
            out_file.write(response.read())
        with zipfile.ZipFile(filename, "r") as zip_ref:
            zip_ref.extractall()
        os.remove(filename)
        print("Download complete!")
    else:
        print("Dataset already exists.")

def preprocess_sentence(sentence):
    """Lowercase, add spaces around punctuation, and add <start>/<end> tokens."""
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = sentence.strip()
    return "<start> " + sentence + " <end>"

def load_dataset(num_examples=15000):
    """Load and preprocess English-German pairs, filtered by MAX_LENGTH."""
    download_data()
    with open("deu.txt", "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
    pairs = []
    for line in lines[:num_examples]:
        parts = line.split("\t")
        if len(parts) >= 2:
            eng, deu = preprocess_sentence(parts[0]), preprocess_sentence(parts[1])
            if len(eng.split()) <= MAX_LENGTH and len(deu.split()) <= MAX_LENGTH:
                pairs.append([eng, deu])
    print(f"Loaded {len(pairs)} sentence pairs")
    return zip(*pairs)

input_texts, target_texts = load_dataset(num_examples=15000)
input_texts, target_texts = list(input_texts), list(target_texts)

MAX_VOCAB_SIZE = 10000
input_tokenizer = keras.preprocessing.text.Tokenizer(
    num_words=MAX_VOCAB_SIZE, filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n', oov_token="<UNK>")
target_tokenizer = keras.preprocessing.text.Tokenizer(
    num_words=MAX_VOCAB_SIZE, filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n', oov_token="<UNK>")
input_tokenizer.fit_on_texts(input_texts)
target_tokenizer.fit_on_texts(target_texts)

input_sequences = keras.preprocessing.sequence.pad_sequences(
    input_tokenizer.texts_to_sequences(input_texts), maxlen=MAX_LENGTH, padding="post")
target_sequences = keras.preprocessing.sequence.pad_sequences(
    target_tokenizer.texts_to_sequences(target_texts), maxlen=MAX_LENGTH, padding="post")

input_vocab_size = min(MAX_VOCAB_SIZE, len(input_tokenizer.word_index)) + 1
target_vocab_size = min(MAX_VOCAB_SIZE, len(target_tokenizer.word_index)) + 1
print(f"Input vocabulary: {input_vocab_size}  |  Target vocabulary: {target_vocab_size}")

split_idx = int(0.8 * len(input_sequences))
train_input, val_input = input_sequences[:split_idx], input_sequences[split_idx:]
train_target, val_target = target_sequences[:split_idx], target_sequences[split_idx:]
print(f"Training pairs: {len(train_input)}  |  Validation pairs: {len(val_input)}")

## PART 1 — Positional encoding

Attention treats its input as a *set* — without extra information the model cannot tell
the first word from the last. The transformer adds a fixed sinusoidal pattern:

$$PE_{(pos, 2i)} = \sin\!\big(pos / 10000^{2i/d\_model}\big), \qquad
  PE_{(pos, 2i+1)} = \cos\!\big(pos / 10000^{2i/d\_model}\big)$$

In [ ]:
def get_positional_encoding(seq_len, d_model):
    """Create the [1, seq_len, d_model] positional encoding matrix."""
    # Position indices [0 .. seq_len-1] as a column: shape [seq_len, 1] for broadcasting.
    position = np.arange(seq_len)[:, np.newaxis]

    # ___BLANK___ x2: Dimension indices for the '2i' term: [0, 2, 4, ..., d_model-2].
    # Hint: an arange from 0 up to d_model with a step of 2.
    div_term = np.arange(0, ___BLANK___, ___BLANK___)  # step size 2: we want the even indices 2i

    # ___BLANK___ x2: The scaling factor 1 / 10000^(2i / d_model).
    # Hint: np.power(base, exponent) with base 10000 and exponent 2i/d_model.
    div_term = 1 / np.___BLANK___(10000, ___BLANK___ / d_model)

    pos_encoding = np.zeros((seq_len, d_model))

    # ___BLANK___ x2: sine on the EVEN columns, cosine on the ODD columns.
    pos_encoding[:, 0::2] = np.___BLANK___(position * div_term)
    pos_encoding[:, 1::2] = np.___BLANK___(position * div_term)

    return tf.cast(pos_encoding[np.newaxis, ...], dtype=tf.float32)  # [1, seq_len, d_model]

# Visual check (provided): you should see horizontal stripes of different frequencies.
import matplotlib.pyplot as plt
pe = get_positional_encoding(MAX_LENGTH, D_MODEL)
plt.figure(figsize=(9, 3))
plt.imshow(pe[0].numpy().T, aspect="auto", cmap="RdBu")
plt.xlabel("position"); plt.ylabel("dimension"); plt.title("Positional encoding")
plt.colorbar(); plt.tight_layout(); plt.show()

## PART 2 — Scaled dot-product attention

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + \text{mask}\right)V$$

In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    query/key/value: [batch, num_heads, seq_len, depth]
    mask: 1 where positions must be HIDDEN (padding or future tokens), broadcastable.
    Returns (output, attention_weights).
    """
    # ___BLANK___ x3: The similarity scores Q @ K^T.
    # Hint: tf.matmul with transpose_b=True computes the product against K transposed.
    matmul_qk = tf.matmul(___BLANK___, ___BLANK___, transpose_b=___BLANK___)

    # Scale by sqrt(d_k) so the softmax keeps useful gradients.
    dk = tf.cast(tf.shape(key)[-1], tf.float32)
    scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

    # Masked positions get a huge negative logit -> ~0 weight after softmax.
    if mask is not None:
        scaled_attention_logits += (mask * -1e9)

    # ___BLANK___ x2: Softmax so each query's weights over the KEY positions sum to 1.
    # Hint: the key dimension is the LAST axis.
    attention_weights = tf.nn.___BLANK___(scaled_attention_logits, axis=___BLANK___)

    output = tf.matmul(attention_weights, value)   # weighted sum of the values
    return output, attention_weights

# Self-test (provided) — run it!
q = k = tf.eye(4, batch_shape=[1, 1]) * 10
v = tf.reshape(tf.range(16, dtype=tf.float32), (1, 1, 4, 4))
out, w = scaled_dot_product_attention(q, k, v)
assert out.shape == (1, 1, 4, 4) and w.shape == (1, 1, 4, 4)
assert np.allclose(tf.reduce_sum(w, -1).numpy(), 1.0), "each row of weights must sum to 1"
assert (tf.argmax(w[0, 0], -1).numpy() == np.arange(4)).all(), "each query should match its own key"
print("scaled dot-product attention self-test passed ✔")

## PART 3 — Multi-head attention *(provided — read it!)*

One head can only focus on one relationship at a time. Multi-head attention projects
Q/K/V, splits `d_model` into `num_heads` slices of size `depth`, attends in parallel,
then concatenates and projects back.

In [ ]:
class MultiHeadAttention(layers.Layer):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % num_heads == 0        # heads must divide the model dimension
        self.depth = d_model // num_heads      # per-head dimension

        self.wq = layers.Dense(d_model)        # Q projection
        self.wk = layers.Dense(d_model)        # K projection
        self.wv = layers.Dense(d_model)        # V projection
        self.dense = layers.Dense(d_model)     # output projection after concatenation

    def split_heads(self, x, batch_size):
        """[batch, seq, d_model] -> [batch, num_heads, seq, depth]"""
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, v, k, q, mask=None):
        batch_size = tf.shape(q)[0]

        q = self.split_heads(self.wq(q), batch_size)   # [batch, heads, seq_q, depth]
        k = self.split_heads(self.wk(k), batch_size)
        v = self.split_heads(self.wv(v), batch_size)

        attention, attention_weights = scaled_dot_product_attention(q, k, v, mask)

        attention = tf.transpose(attention, perm=[0, 2, 1, 3])          # [batch, seq_q, heads, depth]
        concat = tf.reshape(attention, (batch_size, -1, self.d_model))  # merge heads
        return self.dense(concat), attention_weights

## PART 4 — Feed-forward network *(provided)*

Applied independently at every position: `Dense(d_ff, relu) -> Dense(d_model)`.

In [ ]:
class FeedForwardNetwork(layers.Layer):
    def __init__(self, d_model, d_ff, dropout_rate=0.1):
        super(FeedForwardNetwork, self).__init__()
        self.dense1 = layers.Dense(d_ff, activation="relu")  # expand to d_ff
        self.dense2 = layers.Dense(d_model)                  # project back to d_model
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        x = self.dense1(x)
        x = self.dropout(x, training=training)
        return self.dense2(x)

## PART 5 — Encoder layer

Self-attention and FFN, each wrapped in the transformer's signature pattern:
**residual connection + layer normalization** — `LayerNorm(x + Sublayer(x))`.

In [ ]:
class EncoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout_rate=0.1):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForwardNetwork(d_model, d_ff, dropout_rate)
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, x, mask, training=False):
        # Sub-layer 1: multi-head SELF-attention (Q = K = V = x).
        attn_output, _ = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output, training=training)

        # ___BLANK___: Residual connection, then LayerNorm.
        # Hint: normalize the SUM of the sub-layer's input and its output.
        out1 = self.layernorm1(___BLANK___)

        # Sub-layer 2: position-wise feed-forward network.
        ffn_output = self.ffn(out1, training=training)
        ffn_output = self.dropout2(ffn_output, training=training)

        # ___BLANK___: Same pattern for the second sub-layer.
        out2 = self.layernorm2(___BLANK___)
        return out2

## PART 6 — Decoder layer *(provided — read it!)*

Three sub-layers: **masked self-attention** (can't see the future), **cross-attention**
(queries from the decoder, keys/values from the encoder — Lab 5's attention, reborn),
and the FFN. Residual + LayerNorm around each.

In [ ]:
class DecoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout_rate=0.1):
        super(DecoderLayer, self).__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)  # masked self-attention
        self.mha2 = MultiHeadAttention(d_model, num_heads)  # cross-attention to encoder
        self.ffn = FeedForwardNetwork(d_model, d_ff, dropout_rate)
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm3 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)
        self.dropout3 = layers.Dropout(dropout_rate)

    def call(self, x, encoder_output, look_ahead_mask, padding_mask, training=False):
        # 1) Masked self-attention over the target sequence.
        attn1, attn_weights1 = self.mha1(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.layernorm1(x + attn1)

        # 2) Cross-attention: decoder queries attend over encoder keys/values.
        attn2, attn_weights2 = self.mha2(encoder_output, encoder_output, out1, padding_mask)
        attn2 = self.dropout2(attn2, training=training)
        out2 = self.layernorm2(out1 + attn2)

        # 3) Feed-forward network.
        ffn_output = self.ffn(out2, training=training)
        ffn_output = self.dropout3(ffn_output, training=training)
        out3 = self.layernorm3(out2 + ffn_output)
        return out3, attn_weights1, attn_weights2

## PART 7 — Encoder and decoder stacks *(provided)*

In [ ]:
class Encoder(layers.Layer):
    """Embedding + positional encoding + a stack of EncoderLayers."""
    def __init__(self, num_layers, d_model, num_heads, d_ff, vocab_size, max_len, dropout_rate=0.1):
        super(Encoder, self).__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.embedding = layers.Embedding(vocab_size, d_model)
        self.pos_encoding = get_positional_encoding(max_len, d_model)
        self.enc_layers = [EncoderLayer(d_model, num_heads, d_ff, dropout_rate)
                           for _ in range(num_layers)]
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, x, mask, training=False):
        seq_len = tf.shape(x)[1]
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))  # scale, per the paper
        x += self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)
        for layer in self.enc_layers:
            x = layer(x, mask, training=training)
        return x

class Decoder(layers.Layer):
    """Embedding + positional encoding + a stack of DecoderLayers."""
    def __init__(self, num_layers, d_model, num_heads, d_ff, vocab_size, max_len, dropout_rate=0.1):
        super(Decoder, self).__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.embedding = layers.Embedding(vocab_size, d_model)
        self.pos_encoding = get_positional_encoding(max_len, d_model)
        self.dec_layers = [DecoderLayer(d_model, num_heads, d_ff, dropout_rate)
                           for _ in range(num_layers)]
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, x, encoder_output, look_ahead_mask, padding_mask, training=False):
        seq_len = tf.shape(x)[1]
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x += self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)
        for layer in self.dec_layers:
            x, _, _ = layer(x, encoder_output, look_ahead_mask, padding_mask, training=training)
        return x

## PART 8 — The full transformer *(provided — read the masks!)*

Two kinds of mask (1 = hide): a **padding mask** so attention ignores the zeros, and a
**look-ahead mask** so each decoder position sees only earlier positions.

In [ ]:
class Transformer(keras.Model):
    def __init__(self, num_layers, d_model, num_heads, d_ff,
                 input_vocab_size, target_vocab_size, max_len, dropout_rate=0.1):
        super(Transformer, self).__init__()
        self.encoder = Encoder(num_layers, d_model, num_heads, d_ff,
                               input_vocab_size, max_len, dropout_rate)
        self.decoder = Decoder(num_layers, d_model, num_heads, d_ff,
                               target_vocab_size, max_len, dropout_rate)
        self.final_layer = layers.Dense(target_vocab_size)

    def call(self, inputs, training=False):
        inp, tar = inputs

        enc_padding_mask = self.create_padding_mask(inp)         # hide source padding
        dec_padding_mask = self.create_padding_mask(inp)         # for cross-attention
        look_ahead_mask = self.create_look_ahead_mask(tf.shape(tar)[1])   # hide the future
        dec_target_padding_mask = self.create_padding_mask(tar)  # hide target padding
        combined_mask = tf.maximum(dec_target_padding_mask, look_ahead_mask)

        enc_output = self.encoder(inp, enc_padding_mask, training=training)
        dec_output = self.decoder(tar, enc_output, combined_mask, dec_padding_mask,
                                  training=training)
        return self.final_layer(dec_output)   # [batch, tar_len, target_vocab]

    def create_padding_mask(self, seq):
        """1 where seq is padding (id 0). Shape [batch, 1, 1, seq_len] for broadcasting."""
        mask = tf.cast(tf.math.equal(seq, 0), tf.float32)
        return mask[:, tf.newaxis, tf.newaxis, :]

    def create_look_ahead_mask(self, size):
        """Upper-triangular 1s: position i may not attend to positions > i."""
        return 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)

## PART 9 — Training setup *(one blank)*

In [ ]:
print("\n" + "="*70)
print("BUILDING TRANSFORMER MODEL")
print("="*70)

transformer = Transformer(
    num_layers=NUM_LAYERS, d_model=D_MODEL, num_heads=NUM_HEADS, d_ff=D_FF,
    input_vocab_size=input_vocab_size, target_vocab_size=target_vocab_size,
    max_len=MAX_LENGTH, dropout_rate=DROPOUT_RATE)
print("✓ Transformer model created")

# The paper uses a warmup schedule; a fixed rate keeps this lab simple.
# ___BLANK___: Choose the learning rate. Hint: 0.001 is the standard Adam default here.
learning_rate = ___BLANK___

optimizer = keras.optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98, epsilon=1e-9)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

def accuracy_function(real, pred):
    """Token accuracy that ignores padding positions."""
    predictions = tf.cast(tf.argmax(pred, axis=-1), dtype=tf.int32)
    accuracies = tf.equal(predictions, tf.cast(real, tf.int32))
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    accuracies = tf.cast(accuracies, tf.float32) * tf.cast(mask, tf.float32)
    return tf.reduce_sum(accuracies) / tf.reduce_sum(tf.cast(mask, tf.float32))

## PART 10 — Training *(provided)*

Same teacher forcing as Lab 5: decoder input = target shifted right, loss target =
target shifted left, padding masked out of the loss.

In [ ]:
def train_step(inp, tar):
    tar_inp = tar[:, :-1]     # <start> w1 w2 ...
    tar_real = tar[:, 1:]     # w1 w2 ... <end>
    with tf.GradientTape() as tape:
        predictions = transformer([inp, tar_inp], training=True)
        mask = tf.math.not_equal(tar_real, 0)
        loss = loss_fn(tar_real, predictions, sample_weight=mask)
        accuracy = accuracy_function(tar_real, predictions)
    gradients = tape.gradient(loss, transformer.trainable_variables)
    optimizer.apply_gradients(zip(gradients, transformer.trainable_variables))
    return loss, accuracy

def evaluate_model(input_data, target_data):
    total_loss = total_accuracy = 0
    num_batches = len(input_data) // BATCH_SIZE
    for i in range(num_batches):
        inp = input_data[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        tar = target_data[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        tar_inp, tar_real = tar[:, :-1], tar[:, 1:]
        predictions = transformer([inp, tar_inp], training=False)
        mask = tf.math.not_equal(tar_real, 0)
        total_loss += loss_fn(tar_real, predictions, sample_weight=mask).numpy()
        total_accuracy += accuracy_function(tar_real, predictions).numpy()
    return total_loss / num_batches, total_accuracy / num_batches

In [ ]:
print("\n" + "="*70)
print("TRAINING TRANSFORMER")
print("="*70)

best_val_loss, best_val_acc = float("inf"), 0

for epoch in tqdm(range(EPOCHS)):
    indices = np.random.permutation(len(train_input))
    train_input_shuffled = train_input[indices]
    train_target_shuffled = train_target[indices]

    num_batches = len(train_input) // BATCH_SIZE
    total_loss = total_accuracy = 0
    for i in range(num_batches):
        inp = train_input_shuffled[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        tar = train_target_shuffled[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        loss, accuracy = train_step(inp, tar)
        total_loss += loss.numpy()
        total_accuracy += accuracy.numpy()

    train_loss, train_accuracy = total_loss / num_batches, total_accuracy / num_batches
    val_loss, val_accuracy = evaluate_model(val_input, val_target)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {train_loss:.4f}, Acc: {train_accuracy:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")
    if val_loss < best_val_loss:
        best_val_loss, best_val_acc = val_loss, val_accuracy

print(f"\n✓ Training complete!")
print(f"  Best Validation Loss: {best_val_loss:.4f}")
print(f"  Best Validation Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")

## PART 11 — Translation *(provided)*

Greedy decoding: start from `<start>`, repeatedly feed the growing output back in, take
the argmax of the last position, stop at `<end>`.

In [ ]:
def translate_transformer(sentence):
    """Translate an English sentence with the trained transformer."""
    sentence = preprocess_sentence(sentence)
    encoder_input = input_tokenizer.texts_to_sequences([sentence])
    encoder_input = keras.preprocessing.sequence.pad_sequences(
        encoder_input, maxlen=MAX_LENGTH, padding="post")
    encoder_input = tf.convert_to_tensor(encoder_input)

    output = tf.expand_dims([target_tokenizer.word_index["<start>"]], 0)
    for _ in range(MAX_LENGTH):
        predictions = transformer([encoder_input, output], training=False)
        predictions = predictions[:, -1:, :]                 # last position only
        predicted_id = tf.argmax(predictions, axis=-1)
        if predicted_id == target_tokenizer.word_index.get("<end>", 0):
            break
        output = tf.concat([output, tf.cast(predicted_id, tf.int32)], axis=-1)

    result = output.numpy()[0][1:]  # drop <start>
    decoded = [target_tokenizer.index_word.get(i, "") for i in result]
    return " ".join(w for w in decoded if w and w != "<end>")

print("\n" + "="*70)
print("TRANSLATION EXAMPLES")
print("="*70)

test_cases = [
    {"english": "I am a student.", "german_ground_truth": "ich bin ein student"},
    {"english": "How are you?", "german_ground_truth": "wie geht es dir"},
    {"english": "Good morning.", "german_ground_truth": "guten morgen"},
    {"english": "Thank you.", "german_ground_truth": "danke"},
    {"english": "Where is the station?", "german_ground_truth": "wo ist der bahnhof"},
    {"english": "He is running.", "german_ground_truth": "er rennt"},
]
for case in test_cases:
    translation = translate_transformer(case["english"])
    print(f"\nEN: {case['english']}")
    print(f"  reference  : {case['german_ground_truth']}")
    print(f"  transformer: {translation!r}")

## From this lab to BERT and GPT *(provided — read it!)*

| you built              | BERT (Devlin et al.)            | GPT (Brown et al.)               |
|------------------------|---------------------------------|----------------------------------|
| encoder stack          | this IS BERT's architecture     | —                                |
| decoder stack          | —                               | this IS GPT's architecture       |
| look-ahead mask        | not used (bidirectional)        | the reason GPT only looks left   |
| training objective     | masked-token prediction         | next-token prediction            |
| scale                  | 12-24 layers, d_model 768-1024  | up to 96 layers, d_model 12288   |

Same parts, different stacking, vastly more scale and data — plus the risks the
readings discuss: memorization, bias, and the responsibilities of deploying models
trained on the open web.

## Checklist before you submit

* [ ] All cells run top-to-bottom without errors, outputs visible.
* [ ] The positional-encoding plot and the attention self-test pass.
* [ ] Training completes with validation accuracy of at least **~60%** (typically higher).
* [ ] The translation examples produce reasonable German for the simple test sentences.
* [ ] In a final markdown cell (2-4 sentences): map your code to BERT and GPT — which
      stack does each reuse, and which mask makes GPT autoregressive?

**Submit your completed notebook (.ipynb) with all outputs visible.**